<a href="https://colab.research.google.com/github/ChrisP-Bakein6741/Gen_AI2026/blob/main/HW4/Advanced_RAG_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE')


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
file_names = [
    "alchemist.pdf",
    "jayz.pdf",
    "jcole.pdf",
    "nprRap.pdf",
    "pharrell.pdf"
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing alchemist.pdf...
Processing jayz.pdf...


Processing jcole.pdf...
Processing nprRap.pdf...


Processing pharrell.pdf...

Success! Total text chunks created: 85


In [12]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)



# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store



# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy




# Finalize Retrievers
if vectorstore:
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 5

    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    print("\n Hybrid Retriever is ready!")



Embedding 85 chunks with rate-limit protection...
 Processed 15/85...
 Processed 30/85...
 Processed 45/85...
 Processed 60/85...
 Processed 75/85...
 Processed 85/85...

 Hybrid Retriever is ready!


In [13]:
# --- ASSIGNMENT TRIALS: RETRIEVER DEFINITIONS ---

# Trial A: Naive Similarity (k=5)
naive_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Trial B: MMR - Maximal Marginal Relevance (k=5, fetch_k=20)
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 5, 'fetch_k': 20}
)

# Trial C: Hybrid (Keyword + MMR)
# This combines the BM25 keyword search with the MMR diversity search.
hybrid_retriever = EnsembleRetriever(
    retrievers=[mmr_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

print("Retriever Trials A, B, and C are configured.")

Retriever Trials A, B, and C are configured.


In [17]:
# UPDATED 2026 IMPORTS
try:
    # Most stable 2026 path for Contextual Compression
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors.flashrank_rerank import FlashrankRerank
except ImportError:
    # Alternative path used in some v1.x sub-versions
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank



# Initialize FlashRank (The tiny but mighty re-ranker)
# We set top_n=3 to ensure the LLM only sees the most relevant facts.
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)



# Create the Compression Retriever
# This wraps the Hybrid Retriever we just successfully built.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)



#  Final Test of the Pipeline
query = "What were the names of Einstein's children?"
final_docs = compression_retriever.invoke(query)

print(f" Re-ranker initialized and tested. Retrieved {len(final_docs)} chunks.")



# Display the high-quality results
for i, doc in enumerate(final_docs):
    print(f"\nRANKED RESULT {i+1} (Source: {doc.metadata.get('source')}):")
    print(f"{doc.page_content[:450]}...")



ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 31.9MiB/s]


 Re-ranker initialized and tested. Retrieved 3 chunks.

RANKED RESULT 1 (Source: alchemist.pdf):
Born
25 October 1977 (age 48)
Born In
Beverly Hills, Los Angeles County, California,
United States
Biography
Alan Daniel Maman (born October 25, 1977), better known as The
Alchemist, is a hip-hop producer and MC. The Alchemist has
been an active producer since 1991. He hails from affluent
Beverly Hills, California. After rising to prominence in the late
1990s as a close associate of Dilated Peoples, he went on to
produce for many of hip-hop's lea...

RANKED RESULT 2 (Source: jayz.pdf):
Explore Britannica's Newest Newsletter: Money Matters  Learn More
Games & QuizzesHistory & SocietyScience & TechBiographiesAnimals & NatureGeography & TravelArts & CultureProConMoneyVideos Ask the Chatbot
Load Next Page 
JA
Ÿ
-Z - Empire State
…
Entertainment & Pop CultureMusic, Contemporary GenresRap & Hip-Hop Music



CITE
JAY-ZAmerican rapper and entrepreneur
Also known as: Jay Z, Jay-Z, Shawn Corey

In [18]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub


# Gemini 2.5 Flash is the 'sweet spot' for speed and availability.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1
)



# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)



# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")



# THE LCEL PIPELINE (The "Pipe" Syntax)
rag_chain = (
    RunnableParallel({
        "context": compression_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)



# Final Execution
print("--- Final RAG Project Test (Gemini 2.5 Flash) ---\n")
query = "What were the names of Einstein's children and what happened to his daughter?"

try:
    answer = rag_chain.invoke(query)
    print(f"QUESTION: {query}")
    print(f"\nANSWER:\n{answer}")
except Exception as e:
    print(f" Chain failed: {e}")



--- Final RAG Project Test (Gemini 2.5 Flash) ---

QUESTION: What were the names of Einstein's children and what happened to his daughter?

ANSWER:
I don't know the answer to your question. The provided context does not contain any information about Albert Einstein or his children.


In [20]:
#EXPERIMENTS:

# 1. ESSENTIAL IMPORTS
import os
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 2. MANUALLY DEFINE THE PROMPT (Fixes hub NameError)
# This is the standard RAG prompt used by LangChain
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. DEFINE YOUR THREE TEST QUESTIONS (Factual, Conceptual, Technical)
test_queries = [
    "List three specific career milestones for The Alchemist mentioned in the text.", # Factual
    "How does the document describe the creative evolution of Jay-Z's career?",      # Conceptual
    "What is the needl drop technique?", # Technical
]

# 4. CONFIGURE THE THREE TRIAL RETRIEVERS
# Trial A: Naive Similarity
naive_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# Trial B: MMR (Diversity Search)
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={'k': 5, 'fetch_k': 20})

# Trial C: Hybrid (Keyword + MMR)
# Assuming 'bm25_retriever' was created earlier in your notebook
hybrid_retriever = EnsembleRetriever(
    retrievers=[mmr_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

# 5. EXECUTE THE TRIALS
trial_retrievers = {
    "Trial A (Naive)": naive_retriever,
    "Trial B (MMR)": mmr_retriever,
    "Trial C (Hybrid)": hybrid_retriever
}

print("--- STARTING ASSIGNMENT EXPERIMENTS ---\n")

for trial_name, retriever in trial_retrievers.items():
    print(f"================ {trial_name} ================")

    # Building the chain directly
    # 'llm_with_retry' should be in your memory from the notebook setup
    # If it fails, replace it with 'llm'
    try:
        current_llm = llm_with_retry if 'llm_with_retry' in globals() else llm

        temp_chain = (
            {"context": retriever, "question": RunnablePassthrough()}
            | prompt
            | current_llm
            | StrOutputParser()
        )

        for query in test_queries:
            print(f"\nQUERY: {query}")
            response = temp_chain.invoke(query)
            print(f"ANSWER:\n{response}")

    except NameError as e:
        print(f"Error: {e}. Make sure to run the 'Generation Phase' cell in your notebook first!")
    print("\n")

--- STARTING ASSIGNMENT EXPERIMENTS ---

================ Trial A (Naive) ================

QUERY: List three specific career milestones for The Alchemist mentioned in the text.
ANSWER:
Here are three specific career milestones for The Alchemist mentioned in the text:

1.  He released his long-awaited debut album, "1st Infantry," in 2004.
2.  He was the official tour DJ for Eminem on his 2005 Anger Management 3 Tour.
3.  He was one of the music producers behind the video game Grand Theft Auto: Chinatown Wars.

QUERY: How does the document describe the creative evolution of Jay-Z's career?
ANSWER:
The document describes a specific aspect of Jay-Z's creative evolution related to his marriage to Beyoncé. Their relationship was closely tracked by the media, and in response, they produced "complementary bare-all recordings that helped change the direction of songwriting in hip-hop."

Beyond this, the documents highlight his consistent output of successful albums, such as "Vol. 2: Hard Knock

Analysis

Compare Trial A to Trial B. Did the Naive approach return three chunks that all said the same thing? Did MMR provide a more "complete" picture for your conceptual question?

The naive approach actually did a pretty good job for the factual and technical question but fell short ever so slightly on the conceptual question. It did mention some aspects that were relevant to the question however it did not make Jay-Z's creativity a main focus of its answer the same way that the MMR did with it's approach.

In Trial C, did the Hybrid search find information that the Vector-only searches (A and B) missed? Hint. Look for specific dates, names, or technical IDs in your documents.

The hybrid search did find more accurate and specific data. In trial C there is much more usage of dates then in trials A and B.

If a retriever failed to find the answer, did the LLM admit it didn't know, or did it use its internal knowledge to "guess"? Explain how retrieval quality directly impacts "Groundedness."

In the above trials, the retriever was able to find the answers I was looking for. However while I was testing out some things I noticed there was a time when the retriever was unable to find the answer the LLM did admit that it was not able to answer the question with the given documents. The quality of retrieval directly impacts the response that the LLM is able to give. So a higher quality retrieval will give the LLM the best opportuntity to answer with the most "Groundness." A worse retrieval quality will have a reverse effect on the LLM's repsonse.

Based on your specific dataset, which retrieval strategy would you put into production and why?

I would put the hybrid search into production. I say this because it was able to give the most logical and best fitting answers to each of my 3 questions while providing me with more specific details then the other 2 search options were able to do separately.

Trial  | Scale (1-5) | Explanation
-------|-------------|-----------------------
A      | 2   | This mostly just pulled the first information it found and used it as the answer without really diving deep to find more depth in its responses.
B      | 3| This search was capable of giving a much better and in-depth response for the conceptual question.
C      | 4.5| This search gave the best answers becauce it combined a semantic and lexical search to give the most fitting responses.